# Home Credit 09 — Ranking Routes and Robustness

**目的：** 将 Notebook 08 生成的原子特征证据转换为多条可复核的特征排序路线，为 Notebook 10 的模型驱动式子集搜索提供候选空间。

```text
NB08: Atomic Feature Evidence
          ↓
Route-specific Ranking Rules
          ↓
Tree / Logistic Regression Branches
          ↓
Ranking Sensitivity & Top-k Overlap Diagnostics
          ↓
Formal Ranking Routes
          ↓
NB10: Model-based Subset Search
```

排序路线不是最终特征子集，而是对候选变量的优先级约束。例如，同一条路线可以形成 Top-200、Top-300 等嵌套子集。不同证据侧重 Relevance、Model Gain、Temporal Reliability、Interaction 和 Redundancy，因此使用多种路线以减少对单一启发式评分的依赖。

**本 Notebook 的边界：** 构造与比较特征排序，复用 NB08 已计算的证据，不在这里重新执行最终的模型子集训练。路线上产生的 Top-k 差异与敏感性结果，将用于控制 NB10 的搜索成本。


## 1. 上游证据与排序模块边界

Notebook 8 和 Notebook 9 的职责必须分开。

Notebook 8 问：

> **每个 feature 从不同角度看，分别有什么 Atomic Evidence？**

Notebook 9 问：

> **这些 Evidence 能形成哪些真正不同、值得继续搜索的 priority paths？**

因此本 Notebook不会重新计算：

- IV / MI；
- pairwise redundancy；
- LightGBM Gain / Permutation；
- tree-path interaction clues；
- temporal WOE / Gini evidence。

这些昂贵计算已经属于 Notebook 8。

本 Notebook主要做的是：

```text
读取 Atomic Evidence
        ↓
轻量重排 / 组合
        ↓
Ranking sensitivity
        ↓
Route construction
        ↓
Route de-duplication / overlap audit
```

所以这里故意不读取 Notebook 5 的大训练表，也不训练 candidate subsets。

| Notebook | 核心问题 | 主要计算 |
|---|---|---|
| NB8 | 上游证据内容 | Evidence construction |
| NB9 | 哪些搜索方向值得保留？ | 排序、稳定性、重叠诊断 |
| NB10 | 沿这些方向搜多远、成员是否需要局部修复？ | 模型训练 / wrapper search |
| NB11 | Features 固定后参数怎么调？ | Hyperparameter tuning |

### 为什么这里要做 checkpoint source check？

如果 Notebook 8 重新运行以后 Evidence 已经改变，但 Notebook 9 仍然读取旧的 `selection_table` 或旧 ranking checkpoint，后果会很严重：

> **表面上 NB9 已经重新运行，实际上搜索方向仍建立在旧 Evidence 上。**

因此本版会记录 NB8 ZIP 内关键输出的轻量 fingerprint。

这增加的计算成本几乎可以忽略，但能防止“上游已更新、下游仍读旧缓存”的错误，因此属于应该增加的成本。


In [25]:
# ============================================================
# 0. Colab / Project Setup
# ============================================================

from google.colab import drive
from pathlib import Path
from collections import defaultdict
import zipfile
import json

import numpy as np
import pandas as pd
from IPython.display import display

drive.mount("/content/drive")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/HomeCredit_Project"
)

ARTIFACT_ROOT = (
    PROJECT_ROOT
    / "artifacts"
)

NB8_ZIP = (
    ARTIFACT_ROOT
    / "nb08"
    / "nb08_feature_evidence_outputs.zip"
)

NB9_ROOT = (
    ARTIFACT_ROOT
    / "nb09"
)

CHECKPOINT_DIR = (
    NB9_ROOT
    / "checkpoints"
)

FINAL_DIR = (
    NB9_ROOT
    / "final"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

assert PROJECT_ROOT.exists(), PROJECT_ROOT
assert NB8_ZIP.exists(), NB8_ZIP

print("Project:", PROJECT_ROOT)
print("Notebook 8 ZIP:", NB8_ZIP)
print("Checkpoint:", CHECKPOINT_DIR)
print("Final output:", FINAL_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project: /content/drive/MyDrive/HomeCredit_Project
Notebook 8 ZIP: /content/drive/MyDrive/HomeCredit_Project/artifacts/nb08/nb08_feature_evidence_outputs.zip
Checkpoint: /content/drive/MyDrive/HomeCredit_Project/artifacts/nb09/checkpoints
Final output: /content/drive/MyDrive/HomeCredit_Project/artifacts/nb09/final


In [26]:
# ============================================================
# 1. Read Notebook 8 atomic evidence directly from ZIP
# ============================================================

NB8_MEMBERS = {
    "feature_evidence": (
        "reports/feature_evidence/"
        "feature_evidence_report.csv"
    ),
    "numeric_redundancy": (
        "reports/feature_evidence/"
        "numeric_redundancy_groups.csv"
    ),
    "categorical_redundancy": (
        "reports/feature_evidence/"
        "categorical_redundancy_pairs.csv"
    ),
    "mixed_redundancy": (
        "reports/feature_evidence/"
        "mixed_redundancy_review.csv"
    ),
    "interaction_pairs": (
        "reports/feature_evidence/"
        "interaction_pair_evidence.csv"
    ),
    "interaction_features": (
        "reports/feature_evidence/"
        "interaction_feature_evidence.csv"
    ),
    "temporal_watchlist": (
        "reports/feature_evidence/"
        "temporal_feature_watchlist.csv"
    ),
    "config": (
        "reports/feature_evidence/"
        "nb8_feature_evidence_config.json"
    ),
}


def read_csv_from_zip(
    zip_path,
    internal_path,
):
    with zipfile.ZipFile(
        zip_path,
        "r",
    ) as zip_file:
        with zip_file.open(
            internal_path
        ) as file:
            return pd.read_csv(file)


def read_json_from_zip(
    zip_path,
    internal_path,
):
    with zipfile.ZipFile(
        zip_path,
        "r",
    ) as zip_file:
        with zip_file.open(
            internal_path
        ) as file:
            return json.load(file)


def nb8_source_fingerprint(
    zip_path,
    member_paths,
):
    """
    使用 ZIP central directory 中的 CRC + file size
    生成轻量 fingerprint。

    不重新读取完整数据文件做 hashing，
    但只要 NB8 的关键输出内容改变，
    ZIP member CRC 通常就会改变。
    """
    parts = []

    with zipfile.ZipFile(
        zip_path,
        "r",
    ) as zip_file:

        available = set(
            zip_file.namelist()
        )

        for internal_path in sorted(
            set(member_paths)
        ):
            if internal_path not in available:
                raise FileNotFoundError(
                    "NB8 ZIP 缺少："
                    f"{internal_path}"
                )

            info = zip_file.getinfo(
                internal_path
            )

            parts.append(
                "|".join([
                    internal_path,
                    str(info.CRC),
                    str(info.file_size),
                ])
            )

    return "||".join(parts)


feature_evidence = read_csv_from_zip(
    NB8_ZIP,
    NB8_MEMBERS["feature_evidence"],
)

numeric_redundancy = read_csv_from_zip(
    NB8_ZIP,
    NB8_MEMBERS["numeric_redundancy"],
)

categorical_redundancy_pairs = read_csv_from_zip(
    NB8_ZIP,
    NB8_MEMBERS["categorical_redundancy"],
)

mixed_redundancy_review = read_csv_from_zip(
    NB8_ZIP,
    NB8_MEMBERS["mixed_redundancy"],
)

interaction_pair_evidence = read_csv_from_zip(
    NB8_ZIP,
    NB8_MEMBERS["interaction_pairs"],
)

interaction_feature_evidence = read_csv_from_zip(
    NB8_ZIP,
    NB8_MEMBERS["interaction_features"],
)

temporal_watchlist = read_csv_from_zip(
    NB8_ZIP,
    NB8_MEMBERS["temporal_watchlist"],
)

nb8_config = read_json_from_zip(
    NB8_ZIP,
    NB8_MEMBERS["config"],
)

NB8_SOURCE_FINGERPRINT = (
    nb8_source_fingerprint(
        NB8_ZIP,
        NB8_MEMBERS.values(),
    )
)

print(
    "Feature evidence:",
    feature_evidence.shape,
)

print(
    "Notebook 8 candidate features:",
    nb8_config.get(
        "candidate_feature_count"
    ),
)

print(
    "NB8 methodology role:",
    nb8_config.get(
        "methodology_role",
        "legacy NB8 output",
    ),
)

print(
    "NB8 source fingerprint:",
    NB8_SOURCE_FINGERPRINT[:120],
    "...",
)

display(
    feature_evidence.head(10)
)

Feature evidence: (1209, 38)
Notebook 8 candidate features: 1209
NB8 methodology role: legacy NB8 output
NB8 source fingerprint: reports/feature_evidence/categorical_redundancy_pairs.csv|3458066811|1516||reports/feature_evidence/feature_evidence_rep ...


,feature,dtype,dtype_family,role,origin,missing_rate,valid_count,sample_unique_count,sample_percent_unique,sample_dominant_share,...,interaction_anchor_count,interaction_anchor_examples,interaction_rescue_candidate,temporal_iv_mean,temporal_iv_cv,temporal_gini_min,temporal_gini_slope,latest_woe_profile_corr,max_woe_sign_flip_share,temporal_label
0,datelastinstal40dpd_247D__days_before_decision,int32,numeric,model_feature,derived_from_date,0.922455,99419,4433,1.7732,0.922612,...,0,NaN,False,0.167447,0.175740,0.094377,0.008819,0.845455,0.000000,STABLE_CANDIDATE
1,pmtcount_693L,double,numeric,model_feature,original,0.885806,146406,30,0.0120,0.885724,...,0,NaN,False,0.022791,1.366895,0.000000,-0.025241,NaN,1.000000,STABLE_CANDIDATE
2,overdueamountmax2date_1142D_min__days_before_d...,int32,numeric,model_feature,derived_from_date,0.663637,431245,3130,1.2520,0.663164,...,0,NaN,False,0.166548,0.142000,0.169292,0.008758,0.954545,0.100000,STABLE_CANDIDATE
3,assignmentdate_238D__days_before_decision,int32,numeric,model_feature,derived_from_date,0.893146,136996,6821,2.7284,0.893272,...,0,NaN,False,0.024947,1.347810,0.000000,-0.025560,NaN,1.000000,STABLE_CANDIDATE
4,numberofoverdueinstlmaxdat_641D_max__days_befo...,int32,numeric,model_feature,derived_from_date,0.666841,427137,2467,0.9868,0.666280,...,0,NaN,False,0.197481,0.104296,0.191324,0.006493,0.954545,0.090909,STABLE_CANDIDATE
5,overdueamountmax2date_1142D_max__days_before_d...,int32,numeric,model_feature,derived_from_date,0.663637,431245,3011,1.2044,0.663164,...,0,NaN,False,0.184443,0.127074,0.181170,0.007788,0.963636,0.090909,STABLE_CANDIDATE
6,dpdmaxdateyear_596T_max,double,numeric,model_feature,original,0.174653,1058163,6,0.0024,0.479904,...,0,NaN,False,0.079788,0.319653,0.081090,-0.021450,-0.200000,0.500000,DIRECTION_REVERSAL_REVIEW
7,lastdelinqdate_224D__days_before_decision,int32,numeric,model_feature,derived_from_date,0.655699,441422,3099,1.2396,0.654940,...,0,NaN,False,0.152805,0.166800,0.162264,0.008269,0.945455,0.000000,STABLE_CANDIDATE
8,dpdmaxdateyear_896T_max,double,numeric,model_feature,original,0.383288,790675,18,0.0072,0.384072,...,0,NaN,False,0.089210,0.314322,0.102277,0.006598,1.000000,0.000000,STABLE_CANDIDATE
9,numberofoverdueinstlmaxdat_641D_min__days_befo...,int32,numeric,model_feature,derived_from_date,0.666841,427137,2551,1.0204,0.666280,...,0,NaN,False,0.179849,0.127698,0.178068,0.004904,0.945455,0.100000,STABLE_CANDIDATE


## 2. 多排序路线的设计动机

如果只有一条 ranking，例如：

```text
A → B → C → D → E → …
```

那么原本接近 $2^{1209}$ 的任意 subset space，会被大幅压缩成一条 nested Top-k 搜索路径。

问题是：

> **搜索空间虽然变小了，但如果这条 ranking 本身有偏差，后面的所有 Top-k 都会继承这个偏差。**

Notebook 8 已经说明，不同 Evidence 回答的不是同一个问题：

```text
Relevance
→ feature 自己和 target 的关系强不强？

Model Evidence
→ learner 在其他 features 同时存在时，是否真正使用它？

Temporal
→ 这种关系跨时间是否可靠？

Interaction
→ 单独弱的 feature，和别的 feature 配合以后是否有价值？

Redundancy
→ 多个高排名 features 是否其实重复表达相似信息？
```

这些 Evidence 不必强行合成一个唯一 `FinalScore`。

原因是，一旦写成：

```text
FinalScore
=
Relevance
+ Model
+ Interaction
+ Temporal
- Redundancy
```

新的核心问题就会变成：

> **这些部分应该分别占多少权重？**

这会把原本真实存在的 trade-off 隐藏进一组人为权重。

所以 Notebook 9 的做法是：

> **用不同 ranking algorithms，把同一张 Evidence Matrix 变成少数几条合理但不同的 search directions。**

六条正式 Tree routes 的具体算法、例子和局限统一放在 **第 8 节**。前面只建立“为什么需要多条 route”的动机，避免重复解释。

### 2.1 为什么不直接做 CMIM / JMI？

Conditional feature selection 的思想很吸引人，因为它想回答：

> 已经有当前 features 后，新 feature 还能带来多少额外信息？

但 Notebook 8 当前保存的是 MI(X;Y)、Spearman、Cramér's V、eta² 和 Tree-path clues，它们并不等于真正的 conditional mutual information。

所以这一版不会把现有统计量包装成一个“万能 Conditional Score”。

如果以后做 CMIM / JMI，应当单独增加真正的 conditional / joint MI computation。


### 2.2 备选方法与搜索复杂度权衡

- **状态式候选管理：** 可以跟踪特征的保留、争议和重新纳入，但需要额外维护一套人工状态转换规则，难以明确复现。
- **Recursive Feature Elimination (RFE)：** 能考虑特征之间的竞争关系，但需要重复训练模型，并可能过度依赖某一种 learner 的重要性估计。
- **少量固定 Top-k 点：** 有利于快速定位大致范围，但单纯选择 300 / 500 / 800 容易忽略局部平台区。

因此当前设计将多路线 ranking 与后续 coarse-to-fine wrapper search 分离：NB09 控制搜索方向，NB10 根据实际模型表现确定深度与局部调整。


## 3. Checkpoints 与计算复用

Colab Runtime 断开后，Python 内存会消失，所以中间结果需要写进 Google Drive。

Notebook 9 的 checkpoint 不需要复杂，只保留两层：

1. `selection_table.parquet`：整理后的 Evidence 主表；
2. 每条 ranking path 一个 CSV，例如 `tree__relevance.csv`、`tree__model_gain.csv`、`lr__relevance.csv`。

这样做主要有两个作用。

第一，Runtime 断掉后，不需要重新生成已经完成的 ranking。

第二，以后新增一条新的 Ranking Perspective 时，已有 paths 直接读取 checkpoint，只生成新的 path。

如果某一条旧 route 的定义改了，只把它加入 `REBUILD_ROUTES` 重新计算即可。

这里的 checkpoint 本质上是在记录：

> **搜索空间已经构造到哪里了。**


In [27]:
# ============================================================
# 2. Checkpoint + Ranking-Sensitivity Settings
# ============================================================

# 手动强制重建开关。
# 正常情况下 NB8 source fingerprint 改变时，
# selection table / routes 会自动失效。
REBUILD_SELECTION_TABLE = False

# 如果只修改某一条正式 route 的定义，
# 把 checkpoint 名称加进来。
REBUILD_ROUTES = set()

# Consensus 的“前部”定义属于 NB9 的 ranking synthesis，
# 不跟 NB10 将来测试的 coarse k 完全绑定。
CONSENSUS_FRONT_LIMIT = 800


# ------------------------------------------------------------
# Sensitivity side-line
# ------------------------------------------------------------
#
# 默认只比较“结构上不同”的定义，
# 而不是做连续 weight grid。
#
# 后续若需要 0.7/0.3 等额外定义，
# 可以直接加到这里；计算仍然很便宜。
#
RELEVANCE_SENSITIVITY_WEIGHTS = {
    # 两端观点：看 IV 与 MI 最多能有多不同。
    "iv_only": {
        "iv_percentile": 1.0,
        "mi_percentile": 0.0,
    },

    # 50/50 附近的小扰动：
    # 用来检查 default weighting 是否脆弱。
    "iv60_mi40": {
        "iv_percentile": 0.6,
        "mi_percentile": 0.4,
    },

    "reference_50_50": {
        "iv_percentile": 0.5,
        "mi_percentile": 0.5,
    },

    "iv40_mi60": {
        "iv_percentile": 0.4,
        "mi_percentile": 0.6,
    },

    "mi_only": {
        "iv_percentile": 0.0,
        "mi_percentile": 1.0,
    },
}

# 与 NB10 coarse grid 对齐，检查不同 ranking
# 在多个搜索深度是否仍然重复。
RANKING_STABILITY_KS = [
    200,
    300,
    400,
    500,
    600,
    700,
    800,
]

# Gain vs Permutation 只在两者都实际存在的 features
# 上比较；Permutation 当前不是全 1209 feature 覆盖。
MODEL_AGREEMENT_KS = [
    25,
    50,
    100,
]


# NB8 的 interaction summary 正好保留了 300 个 pairs，
# 与 INTERACTION_TOP_PAIR_LIMIT=300 相同。
# 这不是“已经证明被截断”，但属于明显的 boundary signal。
# 因为 raw interaction_pair_evidence 已经保存，
# 可以在 NB9 中几乎零模型成本地比较 300 / 500 / 800。
INTERACTION_PAIR_LIMIT_SENSITIVITY = [
    300,
    500,
    800,
]

# Formal interaction route 暂时仍以 300 为 reference。
# 等看完 sensitivity table 后再决定是否改为 500 / 800。
INTERACTION_FORMAL_PAIR_LIMIT = 300

# Temporal label 使用 min_gini < 0 作为其中一个 review boundary。
# 先统计有多少 features 正好靠近 0，
# 再决定是否值得做 threshold sensitivity。
TEMPORAL_GINI_BOUNDARY_BAND = 0.01

# 默认不把 sensitivity variant 自动送进 NB10。
#
# 运行新版 NB8 后先看 diagnostics。
# 只有真正改变 ranking 的 variant，
# 才手动把名字放进下面集合。
#
# 可选值目前：
# {"iv_only", "mi_only"}
PROMOTE_TREE_RELEVANCE_VARIANTS = set()
PROMOTE_LR_RELEVANCE_VARIANTS = set()


## 4. 原子证据和排序规则的接口

到这里，Notebook 9 的整体方法已经确定：

```text
巨大 subset space
        ↓
不做单一 FinalScore / Pure RFE / 全量条件搜索
        ↓
改用少数 Ranking Routes
        ↓
让 Notebook 10 沿不同 Route 搜索 Top-k depth
```

现在真正的问题变成：

> **Notebook 8 已经给了很多 Evidence，怎样把它们安全地转成 Ranking Routes？**

这就是第 4–7 节存在的原因。

---

### 4.1 Notebook 8 给的是“证据原材料”

`feature_evidence_report.csv` 是 Notebook 8 → Notebook 9 的主要 handoff。

例如：

```text
iv_percentile
mi_percentile
gain_percentile
permutation_percentile
temporal_gini_min
max_shared_path_count
missing_rate
```

这些更接近 **Atomic Evidence**：

> 它们描述 feature 本身的某个性质。

而：

```text
relevance_score
model_score
interaction_rescue_candidate
temporal_stable_flag
```

已经带有一定的解释规则：

> “怎样把多个 Atomic Evidence 合起来看？”

所以 Notebook 9 不应该把这些 reference definition 当成唯一真理。

---

### 4.2 为什么不能拿到 Evidence 就直接进入第 8 节排序？

因为有两类问题还没有解决。

第一类：

> **同一份 Evidence，如果换一种合理解释，ranking 是否明显改变？**

例如：

```text
Relevance
= 0.5 × IV + 0.5 × MI
```

只是一个 reference definition。

如果改成：

```text
0.6 × IV + 0.4 × MI
```

Top300 几乎不变，那没有必要增加新的搜索方向。

但如果 Top300 换了很多 feature，就说明：

> Relevance 这个 route family 本身存在真实的 ranking uncertainty。

这属于第 5 节。

第二类：

> **有些 Evidence 根本不是“每个 feature 一个 score”，不能直接 sort。**

最典型就是 Redundancy：

```text
A、B、C 属于同一个 numerical cluster
```

这是 feature 与 feature 之间的关系，不是：

```text
A = 0.8
B = 0.7
C = 0.6
```

所以必须先转换成排序算法能够使用的结构。

这属于第 6 节。

---

### 4.3 六类 NB8 Evidence 到了 NB9 分别去哪？

| NB8 Evidence | NB8 已经完成什么 | NB9 还需要做什么 | 最后怎样影响 Route |
|---|---|---|---|
| Coverage | missing / unique / dominant 等画像与 flag | 当前不额外生成 Coverage ranking；继续作为 context 保存 | 不单独形成 Route |
| Relevance | IV、MI | 检查不同 IV/MI 解释是否改变 ranking | Relevance Route / 必要时少量 variant |
| Model Contribution | Gain、Permutation | 检查两种 model evidence 是否明显 disagreement | 正式 Model Gain Route；Permutation 主要做诊断 |
| Redundancy | numerical cluster、categorical/mixed pairs | 先转成 cluster / adjacency 结构 | Redundancy-aware Route |
| Interaction | raw tree-path pair clues | 检查 rescue 定义和 pair cap 是否改变 ranking | Interaction Route |
| Temporal | period Gini / WOE / stability evidence | 检查人工 boundary 是否会让 ranking 很脆弱 | Temporal Route |

注意：

> **不是每一种 Evidence 都必须做同一种“敏感性分析”。**

Notebook 9 只做那些**会直接影响 search direction**、而且能利用 NB8 已保存结果廉价检查的事情。

所以第 4–7 节不是新的 Evidence Profiling。

它们是一座桥：

```text
NB8 Atomic Evidence
        ↓
第 4 节：明确哪些是原材料、哪些已经带解释
        ↓
第 5 节：检查 Ranking Definition 是否脆弱
        ↓
第 6 节：把关系型 Evidence 转成算法输入
        ↓
第 7 节：准备通用 Ranking Helpers
        ↓
第 8 节：正式生成六种 Ranking Algorithms
```

`selection_table` 在这里仍然只是共同 Evidence Map，不是最终 feature selection 结果。


In [28]:
# ============================================================
# 3. Evidence Matrix Checkpoint
#    + automatic upstream cache invalidation
# ============================================================

SELECTION_TABLE_PATH = (
    CHECKPOINT_DIR
    / "selection_table.parquet"
)

NB8_SOURCE_META_PATH = (
    CHECKPOINT_DIR
    / "nb8_source_meta.json"
)


cached_source_fingerprint = None

if NB8_SOURCE_META_PATH.exists():

    with open(
        NB8_SOURCE_META_PATH,
        "r",
        encoding="utf-8",
    ) as file:

        cached_source_meta = json.load(
            file
        )

    cached_source_fingerprint = (
        cached_source_meta.get(
            "nb8_source_fingerprint"
        )
    )


NB8_SOURCE_CHANGED = (
    cached_source_fingerprint
    != NB8_SOURCE_FINGERPRINT
)

if NB8_SOURCE_CHANGED:
    print(
        "[INVALIDATE] Notebook 8 evidence changed."
    )
    print(
        "selection_table and ranking routes "
        "will be rebuilt."
    )


EFFECTIVE_REBUILD_SELECTION_TABLE = (
    REBUILD_SELECTION_TABLE
    or NB8_SOURCE_CHANGED
)


if (
    SELECTION_TABLE_PATH.exists()
    and not EFFECTIVE_REBUILD_SELECTION_TABLE
):
    print(
        "[CACHE] selection_table"
    )

    selection_table = pd.read_parquet(
        SELECTION_TABLE_PATH
    )

    SELECTION_TABLE_REBUILT = False

else:
    print(
        "[BUILD] selection_table"
    )

    selection_table = (
        feature_evidence
        .drop_duplicates(
            subset=["feature"]
        )
        .reset_index(drop=True)
        .copy()
    )

    numeric_columns = [
        "missing_rate",
        "iv_percentile",
        "mi_percentile",
        "relevance_score",
        "gain_percentile",
        "permutation_percentile",
        "model_score",
        "numeric_cluster_size",
        "max_shared_path_count",
        "temporal_iv_cv",
        "temporal_gini_min",
        "temporal_gini_slope",
        "latest_woe_profile_corr",
        "max_woe_sign_flip_share",
    ]

    for column in numeric_columns:
        if column in selection_table.columns:
            selection_table[column] = (
                pd.to_numeric(
                    selection_table[column],
                    errors="coerce",
                )
            )

    if (
        "interaction_rescue_candidate"
        in selection_table.columns
    ):
        selection_table[
            "interaction_rescue_candidate"
        ] = (
            selection_table[
                "interaction_rescue_candidate"
            ]
            .fillna(False)
            .astype(bool)
        )

    if (
        "temporal_label"
        in selection_table.columns
    ):
        selection_table[
            "temporal_stable_flag"
        ] = (
            selection_table[
                "temporal_label"
            ]
            .eq("STABLE_CANDIDATE")
        )
    else:
        selection_table[
            "temporal_stable_flag"
        ] = False

    selection_table.to_parquet(
        SELECTION_TABLE_PATH,
        index=False,
    )

    SELECTION_TABLE_REBUILT = True


candidate_features = (
    selection_table["feature"]
    .dropna()
    .drop_duplicates()
    .tolist()
)


# Save current upstream source metadata only after
# selection_table has been successfully loaded / rebuilt.
with open(
    NB8_SOURCE_META_PATH,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        {
            "nb8_source_fingerprint": (
                NB8_SOURCE_FINGERPRINT
            ),
            "candidate_feature_count": (
                len(candidate_features)
            ),
        },
        file,
        ensure_ascii=False,
        indent=2,
    )


# If NB8 changed, all route checkpoints built from the
# old selection table must also be regarded as stale.
FORCE_REBUILD_ROUTES = (
    NB8_SOURCE_CHANGED
    or SELECTION_TABLE_REBUILT
)


print(
    "Candidate features:",
    len(candidate_features),
)

display(
    selection_table[
        [
            column
            for column in [
                "feature",
                "iv_percentile",
                "mi_percentile",
                "relevance_score",
                "gain_percentile",
                "permutation_percentile",
                "interaction_rescue_candidate",
                "max_shared_path_count",
                "temporal_label",
                "temporal_gini_min",
            ]
            if column in selection_table.columns
        ]
    ].head(20)
)

[CACHE] selection_table
Candidate features: 1209


,feature,iv_percentile,mi_percentile,relevance_score,gain_percentile,permutation_percentile,interaction_rescue_candidate,max_shared_path_count,temporal_label,temporal_gini_min
0,datelastinstal40dpd_247D__days_before_decision,0.948718,0.848635,0.898677,0.891646,0.220000,False,0,STABLE_CANDIDATE,0.094377
1,pmtcount_693L,0.830438,0.938792,0.884615,0.275434,NaN,False,0,STABLE_CANDIDATE,0.000000
2,overdueamountmax2date_1142D_min__days_before_d...,0.946237,0.780811,0.863524,0.684036,NaN,False,0,STABLE_CANDIDATE,0.169292
3,assignmentdate_238D__days_before_decision,0.849462,0.874276,0.861869,0.275434,NaN,False,0,STABLE_CANDIDATE,0.000000
4,numberofoverdueinstlmaxdat_641D_max__days_befo...,0.962779,0.754342,0.858561,0.883375,0.340000,False,0,STABLE_CANDIDATE,0.191324
5,overdueamountmax2date_1142D_max__days_before_d...,0.953681,0.758478,0.856079,0.964433,0.820000,False,0,STABLE_CANDIDATE,0.181170
6,dpdmaxdateyear_596T_max,0.934657,0.770885,0.852771,0.275434,NaN,False,0,DIRECTION_REVERSAL_REVIEW,0.081090
7,lastdelinqdate_224D__days_before_decision,0.941274,0.751034,0.846154,0.966088,0.893333,False,0,STABLE_CANDIDATE,0.162264
8,dpdmaxdateyear_896T_max,0.923077,0.768404,0.845740,0.691481,NaN,False,0,STABLE_CANDIDATE,0.102277
9,numberofoverdueinstlmaxdat_641D_min__days_befo...,0.951199,0.739454,0.845327,0.832920,NaN,False,0,STABLE_CANDIDATE,0.178068


## 5. Ranking Definition Sensitivity Audit

本节仍然属于 Notebook 9 的 **副线：Ranking Robustness / Search-Budget Control**。

但它和主线的关系非常直接：

> **只有当一种合理的 Evidence 解释改变 ranking，才值得把这种差异继续传给昂贵的 Notebook 10。**

它不重新读取 Train，也不重新计算 IV、MI、Gain、Temporal Gini。

换句话说：

```text
NB8
→ 检查 Evidence 本身怎么计算、算得是否可靠

NB9 第5节
→ 接受已经算好的 Evidence
→ 只改变“如何解释它”
→ 看 ranking / Top-k 是否变化
```

---

### 一个简化示例：为什么这件事属于 NB9？

假设 Notebook 8 已经给出：

```text
Feature A:
IV = 0.90
MI = 0.40

Feature B:
IV = 0.60
MI = 0.80
```

这些 Evidence 数字现在不重算。

如果 NB9 使用：

```text
50% IV + 50% MI
```

可能得到：

```text
B > A
```

而使用：

```text
60% IV + 40% MI
```

可能得到：

```text
A > B
```

这里改变的不是 Evidence。

改变的是：

> **Evidence → Ranking 的规则。**

所以这属于 Notebook 9。

---

### 5.1 Relevance：reference weighting 是否把搜索方向锁死？

Relevance 的原材料是：

```text
IV percentile
MI percentile
```

reference definition 是：

```text
50/50
```

NB9 额外比较：

```text
IV-only
60/40
50/50
40/60
MI-only
```

目的不是寻找：

```text
IV = 0.63
MI = 0.37
```

这种看起来很精确的“最佳权重”。

而是回答：

> **不同合理观点，是否形成不同 Ranking？**

如果：

```text
60/40
50/50
40/60
```

Top-k 几乎一样：

> 50/50 不脆弱，不需要扩大 NB10 搜索空间。

如果连这种小扰动都换了很多 feature：

> Relevance family 可能存在真实的 ranking uncertainty。

这时才有理由在后面 Promote 少量有代表性的 variant。

---

### 5.2 Model Contribution：Gain 与 Permutation 的 disagreement 有没有搜索意义？

Notebook 8 已经算好了：

```text
Gain
Permutation Importance
```

NB9 不重新训练模型。

这里只问：

> **Tree 训练时大量使用的 feature，和 assessment-side model reliance 是否一致？**

还有一个现实限制：

```text
Gain
→ 可以覆盖完整 candidate pool

Permutation
→ 当前只覆盖 Gain 前部的一部分 features
```

所以不能把 Permutation 假装成完整 1,209-feature Route。

这里只在共同覆盖区域检查：

```text
rank correlation
Top-k overlap
rank gap
```

如果两者基本一致：

> Gain 作为正式 Tree route 的主要 model evidence 足够有代表性。

如果明显 disagreement：

> 先把这种不确定性记录下来，而不是自动创造一条残缺的 Permutation Route。

---

### 5.3 Interaction：不是重算 pair，而是检查“谁需要被 rescue”

Notebook 8 已经保存：

```text
raw tree-path pair evidence
```

Notebook 9 关心的是：

> **这些 pair clues 最终怎样改变 feature ranking？**

因此这里做两件便宜检查。

第一：

```text
旧 rescue
→ exclusive rescue
```

避免把已经被 Model / Relevance Evidence 强力支持的 feature，再重复叫成“靠 Interaction 才救回来”。

第二：

```text
Top300 pairs
Top500 pairs
Top800 pairs
```

比较 rescue set 是否还在明显变化。

如果 300 / 500 / 800 几乎一样：

> 300 已经足够，不扩大后续成本。

如果 500 / 800 继续带来很多真正新的 rescue features：

> Interaction Route 的当前人工边界可能太紧。

这里检查的是：

> **pair evidence 怎样影响 Route membership。**

不是重新计算 tree-path evidence 本身。

---

### 5.4 Temporal：检查排序规则是否卡在人工 boundary 上

Notebook 8 已经算好了不同时间段的：

```text
Gini / WOE / stability evidence
```

Temporal Route 后面会优先考虑稳定 feature。

但如果一个规则包含：

```text
temporal_gini_min < 0
```

那么：

```text
-0.002
```

和：

```text
+0.002
```

可能被放到不同一边。

因此 NB9 先问：

> **有多少 feature 挤在 0 附近？**

如果只有很少：

> 这个 boundary 对 ranking 影响有限，不值得继续扩展 sensitivity。

如果很多：

> Temporal Route 可能对人工 threshold 很敏感，需要进一步审查。

所以这也是：

> **Route-definition robustness**

而不是重新做 Temporal Evidence Profiling。

---

### 5.5 为什么 Coverage 没有在这里单独做一组 weighting sensitivity？

Coverage 并没有消失。

例如：

```text
missing_rate
```

仍然保留在 Evidence Matrix 中。

但当前 NB9 没有定义：

```text
Coverage Score
→ 直接决定完整 ranking
```

也没有写：

```text
FinalScore
=
Relevance × CoveragePenalty
```

所以目前没有一个 Coverage weighting 需要在 NB9 做：

```text
0.5 / 0.6 / 0.7
```

这样的 Ranking sensitivity。

换句话说：

> **Coverage 目前是 context / flag，而不是独立 ranking operator。**

以后如果设计：

```text
Coverage-aware Route
```

那它的 ranking rule 才需要进入这一节做 robustness audit。

---

### 5.6 为什么 Redundancy 没有和前面四种一起写成普通 sensitivity？

因为 Redundancy 的问题不是：

```text
某个 score 的权重该用多少？
```

而是它的数据形态不同：

```text
A 与 B 高度重复
A、B、C 属于同一 cluster
```

这是**关系型 Evidence**。

它必须先被转换成：

```text
feature → numerical cluster
feature → categorical/mixed neighbors
```

第 8 节的 sequential re-ranking 才能使用。

所以它不属于这一节的普通 scalar ranking sensitivity。

它下一步进入：

> **第 6 节：Evidence Representation / Algorithm Preparation**

---

### 5.7 副线的成本原则：为什么值得做这些检查？

这一节主要做：

```text
重新组合已有数字
排序
rank correlation
Jaccard / membership changes
boundary counts
```

成本远低于一次真正的 subset model fit。

但它可以提前回答：

> **这个“不同定义”值得不值得让 Notebook 10 再训练一整条 Route × Depth？**

所以成本原则是：

```text
便宜阶段
→ 允许检查多个合理定义

如果差异只是名字不同
→ 不传给 NB10

如果改变搜索区域
→ 才保留少量代表性 variant
```

这就是副线和主线的连接点。

---


---

### 基于 NB8 已保存证据的诊断依据

本次不是“每个参数都做敏感性分析”。

而是 NB8 的真实结果给出了几个值得检查的信号：

```text
IV vs MI
→ 确实存在明显 disagreement

Gain vs Permutation
→ 共同覆盖区域存在 rank disagreement

Interaction
→ retained pairs 正好碰到人工上限 300

Temporal
→ 已经出现一批 drift / direction-reversal features

Numerical Redundancy
→ 出现多个大型 cluster
```

所以 NB9 只针对这些**可能改变后续搜索方向**的问题付出额外成本。

下一步：

```text
scalar / threshold interpretation 已检查
        ↓
但 Redundancy 仍然不能直接 sort
        ↓
进入第 6 节
```


In [29]:
# ============================================================
# 4. Ranking Sensitivity / Robustness Audit
# ============================================================

def weighted_available_score(
    frame,
    weights,
):
    """
    对非缺失 atomic evidence 做加权平均。
    某个 component 缺失时，只在可用权重上重新归一化。
    """
    numerator = pd.Series(
        0.0,
        index=frame.index,
    )

    denominator = pd.Series(
        0.0,
        index=frame.index,
    )

    for column, weight in weights.items():

        if (
            column not in frame.columns
            or weight == 0
        ):
            continue

        values = pd.to_numeric(
            frame[column],
            errors="coerce",
        )

        available = values.notna()

        numerator = (
            numerator
            + values.fillna(0.0)
            * float(weight)
        )

        denominator = (
            denominator
            + available.astype(float)
            * float(weight)
        )

    return (
        numerator
        / denominator.replace(
            0.0,
            np.nan,
        )
    )


def order_from_score(
    frame,
    score,
):
    temp = frame[
        ["feature"]
    ].copy()

    temp["_score"] = score

    return (
        temp
        .sort_values(
            [
                "_score",
                "feature",
            ],
            ascending=[
                False,
                True,
            ],
            na_position="last",
        )[
            "feature"
        ]
        .tolist()
    )


def rank_map(
    order,
):
    return {
        feature: rank
        for rank, feature
        in enumerate(
            order,
            start=1,
        )
    }


def compare_rankings(
    rankings,
    k_values,
):
    names = list(
        rankings
    )

    rank_maps = {
        name: rank_map(
            rankings[name]
        )
        for name in names
    }

    rows = []

    for index, name_a in enumerate(
        names
    ):
        for name_b in names[
            index + 1:
        ]:

            common_features = [
                feature
                for feature
                in candidate_features
                if (
                    feature in rank_maps[name_a]
                    and feature in rank_maps[name_b]
                )
            ]

            ranks_a = pd.Series(
                [
                    rank_maps[
                        name_a
                    ][feature]
                    for feature
                    in common_features
                ],
                dtype=float,
            )

            ranks_b = pd.Series(
                [
                    rank_maps[
                        name_b
                    ][feature]
                    for feature
                    in common_features
                ],
                dtype=float,
            )

            spearman = (
                ranks_a.corr(
                    ranks_b,
                    method="spearman",
                )
                if len(common_features) > 1
                else np.nan
            )

            mean_abs_rank_shift = float(
                np.mean(
                    np.abs(
                        ranks_a
                        - ranks_b
                    )
                )
            )

            for k in k_values:

                effective_k = min(
                    int(k),
                    len(
                        rankings[name_a]
                    ),
                    len(
                        rankings[name_b]
                    ),
                )

                set_a = set(
                    rankings[
                        name_a
                    ][
                        :effective_k
                    ]
                )

                set_b = set(
                    rankings[
                        name_b
                    ][
                        :effective_k
                    ]
                )

                union = (
                    set_a
                    | set_b
                )

                rows.append({
                    "ranking_a": name_a,
                    "ranking_b": name_b,
                    "k": effective_k,
                    "spearman_rank_corr": spearman,
                    "mean_abs_rank_shift": (
                        mean_abs_rank_shift
                    ),
                    "overlap_count": len(
                        set_a
                        & set_b
                    ),
                    "jaccard": (
                        len(
                            set_a
                            & set_b
                        )
                        / len(union)
                        if union
                        else np.nan
                    ),
                })

    return pd.DataFrame(
        rows
    )


# ------------------------------------------------------------
# Relevance definitions
# ------------------------------------------------------------

relevance_sensitivity_orders = {}

relevance_sensitivity_scores = (
    selection_table[
        ["feature"]
    ].copy()
)

for (
    variant_name,
    weights,
) in RELEVANCE_SENSITIVITY_WEIGHTS.items():

    score = weighted_available_score(
        selection_table,
        weights,
    )

    relevance_sensitivity_scores[
        variant_name
    ] = score

    relevance_sensitivity_orders[
        variant_name
    ] = order_from_score(
        selection_table,
        score,
    )


relevance_sensitivity_summary = (
    compare_rankings(
        relevance_sensitivity_orders,
        [
            k
            for k in RANKING_STABILITY_KS
            if k <= len(candidate_features)
        ],
    )
)

print(
    "Relevance ranking sensitivity"
)

display(
    relevance_sensitivity_summary
)


# ------------------------------------------------------------
# Gain vs Permutation agreement
# only where both are actually observed
# ------------------------------------------------------------

model_common = pd.DataFrame()

if (
    "gain_percentile"
    in selection_table.columns
    and "permutation_percentile"
    in selection_table.columns
):

    model_common = (
        selection_table[
            [
                "feature",
                "gain_percentile",
                "permutation_percentile",
            ]
        ]
        .dropna(
            subset=[
                "gain_percentile",
                "permutation_percentile",
            ]
        )
        .copy()
    )


if len(model_common) > 1:

    model_common[
        "gain_rank"
    ] = (
        model_common[
            "gain_percentile"
        ]
        .rank(
            ascending=False,
            method="average",
        )
    )

    model_common[
        "permutation_rank"
    ] = (
        model_common[
            "permutation_percentile"
        ]
        .rank(
            ascending=False,
            method="average",
        )
    )

    gain_perm_spearman = (
        model_common[
            "gain_rank"
        ]
        .corr(
            model_common[
                "permutation_rank"
            ],
            method="spearman",
        )
    )

    gain_order_common = (
        model_common
        .sort_values(
            "gain_percentile",
            ascending=False,
        )[
            "feature"
        ]
        .tolist()
    )

    permutation_order_common = (
        model_common
        .sort_values(
            "permutation_percentile",
            ascending=False,
        )[
            "feature"
        ]
        .tolist()
    )

    model_agreement_rows = []

    for k in MODEL_AGREEMENT_KS:

        effective_k = min(
            k,
            len(model_common),
        )

        gain_set = set(
            gain_order_common[
                :effective_k
            ]
        )

        perm_set = set(
            permutation_order_common[
                :effective_k
            ]
        )

        union = (
            gain_set
            | perm_set
        )

        model_agreement_rows.append({
            "covered_feature_count": (
                len(model_common)
            ),
            "covered_fraction": (
                len(model_common)
                / len(candidate_features)
            ),
            "k_within_common_coverage": (
                effective_k
            ),
            "spearman_rank_corr": (
                gain_perm_spearman
            ),
            "overlap_count": len(
                gain_set
                & perm_set
            ),
            "jaccard": (
                len(
                    gain_set
                    & perm_set
                )
                / len(union)
                if union
                else np.nan
            ),
        })

    model_gain_permutation_agreement = (
        pd.DataFrame(
            model_agreement_rows
        )
    )

else:

    model_gain_permutation_agreement = (
        pd.DataFrame()
    )


print(
    "Gain / Permutation agreement "
    "within common coverage"
)

display(
    model_gain_permutation_agreement
)

# ------------------------------------------------------------
# Relevance membership changes relative to 50/50
# ------------------------------------------------------------

reference_order = (
    relevance_sensitivity_orders[
        "reference_50_50"
    ]
)

relevance_change_rows = []

for variant_name, variant_order in (
    relevance_sensitivity_orders.items()
):

    if variant_name == "reference_50_50":
        continue

    for k in RANKING_STABILITY_KS:

        if k > len(candidate_features):
            continue

        ref_set = set(
            reference_order[:k]
        )

        alt_set = set(
            variant_order[:k]
        )

        entered = sorted(
            alt_set - ref_set
        )

        exited = sorted(
            ref_set - alt_set
        )

        relevance_change_rows.append({
            "variant": variant_name,
            "k": k,
            "entered_count": len(entered),
            "exited_count": len(exited),
            "entered_examples": "|".join(
                entered[:10]
            ),
            "exited_examples": "|".join(
                exited[:10]
            ),
        })


relevance_membership_changes = (
    pd.DataFrame(
        relevance_change_rows
    )
)

print(
    "Relevance membership changes "
    "relative to 50/50"
)

display(
    relevance_membership_changes
)


# ------------------------------------------------------------
# Gain vs Permutation disagreement diagnostics
# ------------------------------------------------------------

gain_permutation_disagreement = (
    pd.DataFrame()
)

if len(model_common) > 1:

    model_common = (
        model_common
        .copy()
    )

    model_common[
        "rank_gap_gain_minus_perm"
    ] = (
        model_common[
            "gain_rank"
        ]
        - model_common[
            "permutation_rank"
        ]
    )

    model_common[
        "abs_rank_gap"
    ] = (
        model_common[
            "rank_gap_gain_minus_perm"
        ]
        .abs()
    )

    gain_permutation_disagreement = (
        model_common
        .sort_values(
            "abs_rank_gap",
            ascending=False,
        )
        .reset_index(
            drop=True
        )
    )

    print(
        "Largest Gain / Permutation "
        "rank disagreements"
    )

    display(
        gain_permutation_disagreement
        .head(30)
    )


# ------------------------------------------------------------
# Rebuild interaction rescue from raw pair evidence
# ------------------------------------------------------------

reference_relevance_lookup = (
    selection_table
    .set_index("feature")[
        "relevance_score"
    ]
    .to_dict()
)

reference_model_lookup = (
    selection_table
    .set_index("feature")[
        "model_score"
    ]
    .to_dict()
)


def reference_is_strong(
    feature,
):
    relevance = (
        reference_relevance_lookup
        .get(
            feature,
            np.nan,
        )
    )

    model = (
        reference_model_lookup
        .get(
            feature,
            np.nan,
        )
    )

    return (
        (
            pd.notna(relevance)
            and relevance >= 0.80
        )
        or
        (
            pd.notna(model)
            and model >= 0.80
        )
    )


def reference_is_weak_relevance(
    feature,
):
    relevance = (
        reference_relevance_lookup
        .get(
            feature,
            np.nan,
        )
    )

    return (
        pd.notna(relevance)
        and relevance < 0.40
    )


def reference_is_exclusive_rescue_candidate(
    feature,
):
    """
    真正需要 interaction clue 才被保护的候选：
    relevance 弱，而且没有已经被 reference model evidence
    归为 strong anchor。
    """
    return (
        reference_is_weak_relevance(
            feature
        )
        and not reference_is_strong(
            feature
        )
    )


def build_interaction_rescue_table(
    pair_evidence,
    pair_limit,
):
    """
    从 raw tree-path pair evidence 重建更严格的 rescue evidence。
    不使用 NB8 已截断后的 rescue label。
    """
    if pair_evidence.empty:
        return pd.DataFrame({
            "feature": candidate_features,
            "interaction_exclusive_rescue_candidate": False,
            "interaction_exclusive_max_shared_path_count": 0,
            "interaction_exclusive_anchor_count": 0,
        })

    pair_rows = []

    for row in (
        pair_evidence
        .sort_values(
            "shared_path_count",
            ascending=False,
        )
        .itertuples(
            index=False
        )
    ):

        a = row.feature_a
        b = row.feature_b

        a_exclusive = (
            reference_is_exclusive_rescue_candidate(
                a
            )
        )
        b_exclusive = (
            reference_is_exclusive_rescue_candidate(
                b
            )
        )

        a_strong = (
            reference_is_strong(
                a
            )
        )
        b_strong = (
            reference_is_strong(
                b
            )
        )

        qualifies = (
            (
                a_exclusive
                and b_strong
            )
            or
            (
                b_exclusive
                and a_strong
            )
        )

        if qualifies:
            pair_rows.append({
                "feature_a": a,
                "feature_b": b,
                "shared_path_count": (
                    int(
                        row.shared_path_count
                    )
                ),
                "a_exclusive": a_exclusive,
                "b_exclusive": b_exclusive,
                "a_strong": a_strong,
                "b_strong": b_strong,
            })

    top_pairs = (
        pd.DataFrame(
            pair_rows
        )
        .head(
            int(pair_limit)
        )
    )

    stats = {
        feature: {
            "max_shared_path_count": 0,
            "anchors": set(),
        }
        for feature in candidate_features
    }

    for row in (
        top_pairs.itertuples(
            index=False
        )
    ):

        if (
            row.a_exclusive
            and row.b_strong
        ):
            stats[
                row.feature_a
            ][
                "max_shared_path_count"
            ] = max(
                stats[
                    row.feature_a
                ][
                    "max_shared_path_count"
                ],
                int(
                    row.shared_path_count
                ),
            )

            stats[
                row.feature_a
            ][
                "anchors"
            ].add(
                row.feature_b
            )

        if (
            row.b_exclusive
            and row.a_strong
        ):
            stats[
                row.feature_b
            ][
                "max_shared_path_count"
            ] = max(
                stats[
                    row.feature_b
                ][
                    "max_shared_path_count"
                ],
                int(
                    row.shared_path_count
                ),
            )

            stats[
                row.feature_b
            ][
                "anchors"
            ].add(
                row.feature_a
            )

    rows = []

    for feature in candidate_features:

        anchors = (
            stats[
                feature
            ][
                "anchors"
            ]
        )

        rows.append({
            "feature": feature,
            "interaction_exclusive_rescue_candidate": (
                len(anchors) > 0
            ),
            "interaction_exclusive_max_shared_path_count": (
                stats[
                    feature
                ][
                    "max_shared_path_count"
                ]
            ),
            "interaction_exclusive_anchor_count": (
                len(anchors)
            ),
        })

    return pd.DataFrame(
        rows
    )


interaction_limit_tables = {
    pair_limit: (
        build_interaction_rescue_table(
            interaction_pair_evidence,
            pair_limit,
        )
    )
    for pair_limit in (
        INTERACTION_PAIR_LIMIT_SENSITIVITY
    )
}


baseline_interaction_table = (
    interaction_limit_tables[
        INTERACTION_FORMAL_PAIR_LIMIT
    ]
)


# Merge formal interaction evidence into selection_table.
for column in [
    "interaction_exclusive_rescue_candidate",
    "interaction_exclusive_max_shared_path_count",
    "interaction_exclusive_anchor_count",
]:

    if column in selection_table.columns:
        selection_table = (
            selection_table
            .drop(
                columns=[column]
            )
        )


selection_table = (
    selection_table
    .merge(
        baseline_interaction_table,
        on="feature",
        how="left",
        validate="one_to_one",
    )
)


interaction_limit_rows = []

baseline_rescue = set(
    baseline_interaction_table
    .loc[
        baseline_interaction_table[
            "interaction_exclusive_rescue_candidate"
        ],
        "feature",
    ]
)

for pair_limit in (
    INTERACTION_PAIR_LIMIT_SENSITIVITY
):

    table = (
        interaction_limit_tables[
            pair_limit
        ]
    )

    rescue_set = set(
        table.loc[
            table[
                "interaction_exclusive_rescue_candidate"
            ],
            "feature",
        ]
    )

    union = (
        baseline_rescue
        | rescue_set
    )

    interaction_limit_rows.append({
        "pair_limit": pair_limit,
        "exclusive_rescued_feature_count": (
            len(rescue_set)
        ),
        "new_vs_formal_limit": (
            len(
                rescue_set
                - baseline_rescue
            )
        ),
        "lost_vs_formal_limit": (
            len(
                baseline_rescue
                - rescue_set
            )
        ),
        "jaccard_vs_formal_limit": (
            len(
                baseline_rescue
                & rescue_set
            )
            / len(union)
            if union
            else np.nan
        ),
        "new_feature_examples": "|".join(
            sorted(
                rescue_set
                - baseline_rescue
            )[:15]
        ),
    })


interaction_pair_limit_sensitivity = (
    pd.DataFrame(
        interaction_limit_rows
    )
)

print(
    "Exclusive interaction rescue "
    "pair-limit sensitivity"
)

display(
    interaction_pair_limit_sensitivity
)


# How many old NB8 rescues were already model/relevance-strong?
old_rescue_set = set(
    selection_table.loc[
        selection_table[
            "interaction_rescue_candidate"
        ].fillna(False),
        "feature",
    ]
)

exclusive_rescue_set = set(
    selection_table.loc[
        selection_table[
            "interaction_exclusive_rescue_candidate"
        ].fillna(False),
        "feature",
    ]
)

interaction_rescue_recalibration = pd.DataFrame([{
    "nb8_rescue_count": len(
        old_rescue_set
    ),
    "exclusive_rescue_count": len(
        exclusive_rescue_set
    ),
    "old_but_not_exclusive": len(
        old_rescue_set
        - exclusive_rescue_set
    ),
    "exclusive_but_not_old": len(
        exclusive_rescue_set
        - old_rescue_set
    ),
}])

print(
    "Interaction rescue recalibration"
)

display(
    interaction_rescue_recalibration
)


# ------------------------------------------------------------
# Temporal boundary diagnostic
# ------------------------------------------------------------

temporal_boundary_diagnostic = (
    pd.DataFrame()
)

if "temporal_gini_min" in selection_table.columns:

    gini_min = pd.to_numeric(
        selection_table[
            "temporal_gini_min"
        ],
        errors="coerce",
    )

    near_zero = (
        gini_min
        .abs()
        <= TEMPORAL_GINI_BOUNDARY_BAND
    )

    temporal_boundary_diagnostic = (
        pd.DataFrame([{
            "boundary_band": (
                TEMPORAL_GINI_BOUNDARY_BAND
            ),
            "non_missing_count": (
                int(
                    gini_min.notna().sum()
                )
            ),
            "near_zero_count": (
                int(
                    near_zero.sum()
                )
            ),
            "near_zero_fraction": (
                float(
                    near_zero.mean()
                )
            ),
            "negative_within_band": (
                int(
                    (
                        (gini_min < 0)
                        & near_zero
                    ).sum()
                )
            ),
            "nonnegative_within_band": (
                int(
                    (
                        (gini_min >= 0)
                        & near_zero
                    ).sum()
                )
            ),
        }])
    )

print(
    "Temporal min-Gini boundary diagnostic"
)

display(
    temporal_boundary_diagnostic
)


Relevance ranking sensitivity


,ranking_a,ranking_b,k,spearman_rank_corr,mean_abs_rank_shift,overlap_count,jaccard
0,iv_only,iv60_mi40,200,0.854978,153.612903,160,0.666667
1,iv_only,iv60_mi40,300,0.854978,153.612903,237,0.652893
2,iv_only,iv60_mi40,400,0.854978,153.612903,286,0.556420
3,iv_only,iv60_mi40,500,0.854978,153.612903,376,0.602564
4,iv_only,iv60_mi40,600,0.854978,153.612903,490,0.690141
...,...,...,...,...,...,...,...
65,iv40_mi60,mi_only,400,0.841724,161.606286,263,0.489758
66,iv40_mi60,mi_only,500,0.841724,161.606286,400,0.666667
67,iv40_mi60,mi_only,600,0.841724,161.606286,531,0.793722
68,iv40_mi60,mi_only,700,0.841724,161.606286,637,0.834862


Gain / Permutation agreement within common coverage


,covered_feature_count,covered_fraction,k_within_common_coverage,spearman_rank_corr,overlap_count,jaccard
0,150,0.124069,25,0.54598,13,0.351351
1,150,0.124069,50,0.54598,35,0.538462
2,150,0.124069,100,0.54598,74,0.587302


Relevance membership changes relative to 50/50


...

,pair_limit,exclusive_rescued_feature_count,new_vs_formal_limit,lost_vs_formal_limit,jaccard_vs_formal_limit,new_feature_examples
0,300,38,0,0,1.000000,
1,500,52,14,0,0.730769,childnum_21L_mean|clientscnt3m_3712950L|client...
2,800,71,33,0,0.535211,childnum_21L_mean|childnum_21L_std|clientscnt3...


Interaction rescue recalibration


,nb8_rescue_count,exclusive_rescue_count,old_but_not_exclusive,exclusive_but_not_old
0,44,38,18,12


Temporal min-Gini boundary diagnostic


,boundary_band,non_missing_count,near_zero_count,near_zero_fraction,negative_within_band,nonnegative_within_band
0,0.01,1209,547,0.45244,137,410


## 6. Redundancy Evidence 的结构化

第 5 节处理的是：

> **同一批 scalar Evidence，换一种解释是否改变 ranking？**

Redundancy 不太一样。

Notebook 8 给它的不是一个简单分数，而是 feature 与 feature 之间的关系。

例如：

```text
Numerical:
A、B、C → 同一个 redundancy cluster

Categorical / Mixed:
D ↔ E
F ↔ G
```

所以不能像 Relevance 那样直接：

```text
.sort_values("redundancy_score")
```

否则会丢掉最重要的信息：

> **“谁和谁重复？”**

---

### 6.1 这一节真正做什么？

这里只把 Notebook 8 已经算好的 redundancy evidence 整理成第 8 节算法可查询的结构：

```text
Numerical redundancy
→ feature → cluster ID

Categorical / Mixed redundancy
→ feature → redundancy neighbors
```

例如：

```text
A → Cluster 7
B → Cluster 7
C → Cluster 7
```

以及：

```text
D → {E}
E → {D}
```

到这里仍然没有决定：

```text
A 应该排第几？
B 应该延后多少？
```

真正决定排序的是 **8.5 Redundancy-aware Route**。

---

### 6.2 为什么要把这一小步单独留在主线里？

因为它说明了一个很重要的算法差异：

```text
Relevance / Gain / Temporal / Interaction
→ 最后都可以整理成每个 feature 的排序键

Redundancy
→ 必须知道“前面已经选了谁”
```

也就是说，Redundancy-aware 不是普通静态 sort。

它后面会变成：

> **stateful sequential re-ranking**

前面某个 cluster 的成员已经进入前部以后，后面同 cluster feature 的优先级才会发生变化。

所以这里不是多做一次 feature selection。

它只是把：

```text
NB8 的关系型 Evidence
```

转换成：

```text
NB9 Ranking Algorithm 的输入结构
```

下一步再把这些结构交给通用 ranking helpers。


In [30]:
# ============================================================
# 6. Redundancy structures
# ============================================================

# Pair graph is used for categorical / mixed redundancy.
redundancy_graph = defaultdict(set)

# Numerical redundancy is already represented by cluster IDs.
# Do NOT turn a 35-feature cluster into an arbitrary "star"
# centered on whichever row happens to appear first.
numeric_group_lookup = {}


def add_redundancy_edge(
    feature_a,
    feature_b,
):
    if (
        pd.isna(feature_a)
        or pd.isna(feature_b)
        or feature_a == feature_b
    ):
        return

    if (
        feature_a not in candidate_features
        or feature_b not in candidate_features
    ):
        return

    redundancy_graph[
        feature_a
    ].add(
        feature_b
    )

    redundancy_graph[
        feature_b
    ].add(
        feature_a
    )


# ------------------------------------------------------------
# Numerical cluster membership
# ------------------------------------------------------------

if (
    "numeric_redundancy_group"
    in numeric_redundancy.columns
):

    numeric_groups_valid = (
        numeric_redundancy
        .dropna(
            subset=[
                "numeric_redundancy_group"
            ]
        )
        .copy()
    )

    for row in (
        numeric_groups_valid
        .itertuples(
            index=False
        )
    ):
        if (
            row.feature
            in candidate_features
        ):
            numeric_group_lookup[
                row.feature
            ] = (
                row.numeric_redundancy_group
            )


# ------------------------------------------------------------
# Categorical pair redundancy
# ------------------------------------------------------------

for row in (
    categorical_redundancy_pairs
    .itertuples(index=False)
):
    add_redundancy_edge(
        row.feature_a,
        row.feature_b,
    )


# ------------------------------------------------------------
# Mixed pair redundancy
# ------------------------------------------------------------

for row in (
    mixed_redundancy_review
    .itertuples(index=False)
):
    add_redundancy_edge(
        row.numeric_feature,
        row.categorical_feature,
    )


numeric_group_sizes = (
    pd.Series(
        list(
            numeric_group_lookup.values()
        )
    )
    .value_counts()
    .sort_values(
        ascending=False
    )
)


print(
    "Numerical redundancy clusters:",
    int(
        len(
            numeric_group_sizes
        )
    ),
)

print(
    "Largest numerical cluster sizes:",
    numeric_group_sizes
    .head(10)
    .tolist(),
)

print(
    "Features with categorical/mixed "
    "redundancy neighbors:",
    sum(
        len(neighbors) > 0
        for neighbors
        in redundancy_graph.values()
    ),
)


Numerical redundancy clusters: 477
Largest numerical cluster sizes: [35, 34, 34, 31, 21, 19, 19, 16, 14, 12]
Features with categorical/mixed redundancy neighbors: 28


## 7. 排序辅助函数与输入准备

当前主线位置：

```text
NB8 Atomic Evidence
        ↓
第 4 节
分清 Atomic Evidence 与 reference interpretation
        ↓
第 5 节
检查哪些 ranking definition 会改变搜索方向
        ↓
第 6 节
把 Redundancy 这种关系型 Evidence 转成算法输入
        ↓
第 7 节
准备几种通用 ranking operations
        ↓
第 8 节
正式生成六种 Tree Ranking Algorithms
```

这一节本身**不再产生新的 Evidence，也不新增 Route**。

它只把第 8 节会反复使用的操作封装好。

---

### `complete_order()`

作用：

> 保证任何 route 最终都覆盖完整 candidate pool。

设计意义

因为 NB9 的目标是构造：

```text
完整 ranking
```

而不是提前 hard delete features。

这样 Notebook 10 才能继续比较：

```text
Top200
Top300
Top400
...
```

---

### `redundancy_aware_order()`

作用：

> 把第 6 节准备好的 cluster / neighbor 结构真正变成 sequential re-ranking。

它不是再算 Redundancy。

它只是执行：

```text
这个信息组前面已经有代表了吗？
        ↓
有
→ 当前 feature 先 delayed

没有
→ 保留在前部
```

详细算法和例子统一放在 **8.5**。

---

### `consensus_order()`

作用：

> 把前面多条不同 Route 的“排名意见”重新聚合。

它不是把 Evidence 分数再加权。

而是使用：

```text
front votes
median rank
best rank
```

形成一个新的 rank-aggregation perspective。

详细例子放在 **8.6**。

---

### `load_or_build_route()`

作用：

> Route checkpoint / rebuild。

它不改变 ranking 数学逻辑，只保证运行过程可以恢复。

---

所以第 7 节结束时，本阶段已完成所有“准备”。

从下一节开始，问题终于变成：

> **如果使用不同 Ranking Algorithms，同一批 1,209 个 features 会被排成怎样不同的搜索路径？**

这就是第 8 节的主角。


In [31]:
# ============================================================
# 7. Small Ranking Helpers
# ============================================================

def complete_order(
    order,
):
    result = []
    seen = set()

    for feature in order:
        if (
            feature in candidate_features
            and feature not in seen
        ):
            result.append(feature)
            seen.add(feature)

    for feature in candidate_features:
        if feature not in seen:
            result.append(feature)

    return result


def redundancy_aware_order(
    base_order,
):
    selected = []
    selected_set = set()
    selected_numeric_groups = set()
    delayed = []

    for feature in base_order:

        numeric_group = (
            numeric_group_lookup
            .get(
                feature
            )
        )

        numeric_group_conflict = (
            numeric_group is not None
            and numeric_group
            in selected_numeric_groups
        )

        pair_neighbors = (
            redundancy_graph
            .get(
                feature,
                set(),
            )
        )

        pair_conflict = bool(
            pair_neighbors
            & selected_set
        )

        if (
            numeric_group_conflict
            or pair_conflict
        ):
            delayed.append(
                feature
            )

        else:
            selected.append(
                feature
            )

            selected_set.add(
                feature
            )

            if numeric_group is not None:
                selected_numeric_groups.add(
                    numeric_group
                )

    return complete_order(
        [
            *selected,
            *delayed,
        ]
    )


def consensus_order(
    base_routes,
    front_limit,
):
    rank_maps = {
        route_name: {
            feature: rank
            for rank, feature
            in enumerate(
                order,
                start=1,
            )
        }
        for route_name, order
        in base_routes.items()
    }

    rows = []

    for feature in candidate_features:

        ranks = [
            rank_maps[
                route_name
            ][
                feature
            ]
            for route_name
            in base_routes
        ]

        rows.append({
            "feature": feature,
            "front_votes": sum(
                rank <= front_limit
                for rank in ranks
            ),
            "median_rank": float(
                np.median(ranks)
            ),
            "best_rank": int(
                min(ranks)
            ),
        })

    consensus_table = (
        pd.DataFrame(rows)
        .sort_values(
            [
                "front_votes",
                "median_rank",
                "best_rank",
                "feature",
            ],
            ascending=[
                False,
                True,
                True,
                True,
            ],
        )
    )

    return complete_order(
        consensus_table[
            "feature"
        ].tolist()
    )


def load_or_build_route(
    checkpoint_name,
    build_function,
):
    checkpoint_path = (
        CHECKPOINT_DIR
        / f"{checkpoint_name}.csv"
    )

    if (
        checkpoint_path.exists()
        and checkpoint_name
        not in REBUILD_ROUTES
        and not FORCE_REBUILD_ROUTES
    ):
        print(
            f"[CACHE] {checkpoint_name}"
        )

        cached = pd.read_csv(
            checkpoint_path
        )

        return (
            cached["feature"]
            .tolist()
        )

    print(
        f"[BUILD] {checkpoint_name}"
    )

    order = build_function()

    pd.DataFrame({
        "rank": np.arange(
            1,
            len(order) + 1,
        ),
        "feature": order,
    }).to_csv(
        checkpoint_path,
        index=False,
    )

    return order

## 8. Tree Branch — 六条排序路线

前面第 4–7 节解决的是：

```text
Evidence 怎样安全地变成 Ranking Algorithm 的输入？
```

从这里开始，Notebook 9 回到**主线核心**：

> **用不同排序算法，把巨大的 subset space 压缩成少数几条合理但不同的 search directions。**

这里的重点不是“六个 Evidence 名称”。

而是：

> **同一张 Evidence Matrix，被六种不同的排序算法解释以后，会产生六条不同的 feature order。**

每条 Route 都是在回答一个不同的搜索假设：

| Route | 排序算法的核心 | 它优先保护什么 | 主要防的搜索错误 |
|---|---|---|---|
| Relevance | Relevance-first lexicographic ordering | 单独就和 target 强相关的 feature | 单变量强信号被低估 |
| Model Gain | Gain-first lexicographic ordering | Tree 实际大量使用的 feature | 只看单变量、忽略条件利用价值 |
| Temporal Relevance | Stability-first lexicographic ordering | 跨时间更可靠的 feature | 当前很强但未来关系失效 |
| Interaction Relevance | Rescue-first lexicographic ordering | 单独弱、组合后有价值的 feature | interaction feature 被“遗漏” |
| Redundancy-aware | Stateful sequential re-ranking | 信息多样性 | 相似 features 重复占满 Top-k |
| Consensus | Rank aggregation | 多条 route 都稳定支持的 feature | 把全部预算押在单一 heuristic |

这里的 `lexicographic ordering` 可以理解为：

> **排序条件有明确先后级别：先看第一条件；第一条件无法区分时，再看第二条件。**

以下各小节分别展示：

```text
算法规则
→ feature order 为什么改变
→ 这种改变在防什么错误
→ 它自己又有什么坑
```

---

### 8.1 Relevance Route — Relevance-first Lexicographic Ordering

算法顺序：

```text
1. relevance_score 高的优先
2. 再用 gain_percentile 做二级排序
```

它代表的问题是：

> **如果优先采用 feature 自己和 target 的关系，搜索应该从哪里开始？**

### 8.2 Model Gain Route — Gain-first Lexicographic Ordering

算法顺序：

```text
1. gain_percentile 高的优先
2. 再看 relevance_score
```

它代表的问题是：

> **在其他 features 已经同时存在时，Tree learner 实际更愿意使用谁？**

### 8.3 Temporal Relevance Route — Stability-first Lexicographic Ordering

算法顺序：

```text
1. temporal stable 优先
2. temporal_gini_min 更高者优先
3. 再看 relevance_score
```

它代表的问题是：

> **如果更重视未来失效，而不是只追求当前最强预测力，顺序应该怎样变化？**

### 8.4 Interaction Relevance Route — Rescue-first Lexicographic Ordering

算法顺序：

```text
1. exclusive interaction rescue candidate 优先
2. shared-path clue 更强者优先
3. 再看 relevance_score
```

它代表的问题是：

> **有没有 feature 自己看起来弱，但和强 feature 配合时才真正有价值？**

### 8.5 Redundancy-aware Route — Stateful Sequential Re-ranking

这条不是普通的一次性 `sort`。它会从前往后扫描，而且**前面已经选了谁，会影响后面谁被延后**。

算法思想：

```text
按照 base ranking 从前往后扫描
        ↓
同一 numerical cluster 已经有代表？
        ↓ Yes
当前 feature 先 delayed
        ↓ No
保留在前部

最后再把 delayed features 接回完整 ranking
```

它代表的问题是：

> **有限 Top-k 容量里，能不能先放更多不同的信息，而不是很多近似副本？**

### 8.6 Consensus Route — Rank Aggregation

Consensus 不重新计算新的 feature Evidence，而是把前面多条 route 的**排名结果**作为输入。

当前核心顺序是：

```text
1. front_votes：多少条 route 都把它放在前部
2. median_rank：多条 route 的中位排名
3. best_rank：最好的一次排名
```

它代表的问题是：

> **不同算法虽然理由不同，有没有一些 feature 得到跨算法的稳定支持？**

---

### 六条 Route 最重要的区别

可以把它们记成六个不同问题：

```text
Relevance
→ 谁自己最强？

Model Gain
→ Tree 实际最常从谁获得增益？

Temporal
→ 谁跨时间更可靠？

Interaction
→ 谁自己弱、但组合后不能漏？

Redundancy-aware
→ 谁虽然强，但和已经在前部的信息太重复？

Consensus
→ 谁得到多种算法共同支持？
```

所以六条 route 的价值不在于“六个名字”，而在于：

> **六种排序算法故意制造六种合理但不同的 search direction，让 NB10 比较，而不是在 NB9 提前宣布某一种 heuristic 一定正确。**


In [32]:
# ============================================================
# 8. Tree Ranking Routes
# ============================================================

tree_relevance = load_or_build_route(
    "tree__relevance",
    lambda: complete_order(
        selection_table
        .sort_values(
            [
                "relevance_score",
                "gain_percentile",
            ],
            ascending=[
                False,
                False,
            ],
            na_position="last",
        )[
            "feature"
        ]
        .tolist()
    ),
)


tree_gain = load_or_build_route(
    "tree__model_gain",
    lambda: complete_order(
        selection_table
        .sort_values(
            [
                "gain_percentile",
                "relevance_score",
            ],
            ascending=[
                False,
                False,
            ],
            na_position="last",
        )[
            "feature"
        ]
        .tolist()
    ),
)


tree_temporal = load_or_build_route(
    "tree__temporal_relevance",
    lambda: complete_order(
        selection_table
        .sort_values(
            [
                "temporal_stable_flag",
                "temporal_gini_min",
                "relevance_score",
            ],
            ascending=[
                False,
                False,
                False,
            ],
            na_position="last",
        )[
            "feature"
        ]
        .tolist()
    ),
)


tree_interaction = load_or_build_route(
    "tree__interaction_relevance",
    lambda: complete_order(
        selection_table
        .sort_values(
            [
                "interaction_exclusive_rescue_candidate",
                "interaction_exclusive_max_shared_path_count",
                "relevance_score",
            ],
            ascending=[
                False,
                False,
                False,
            ],
            na_position="last",
        )[
            "feature"
        ]
        .tolist()
    ),
)


tree_redundancy = load_or_build_route(
    "tree__redundancy_aware",
    lambda: redundancy_aware_order(
        tree_relevance
    ),
)


tree_base_routes = {
    "relevance": tree_relevance,
    "model_gain": tree_gain,
    "temporal_relevance": tree_temporal,
    "interaction_relevance": tree_interaction,
    "redundancy_aware": tree_redundancy,
}


# Consensus 很便宜，而且依赖“当前有哪些 routes”。
# 所以每次重新生成并覆盖 checkpoint，
# 这样以后新增 route 时它会自动更新。
tree_consensus = consensus_order(
    tree_base_routes,
    CONSENSUS_FRONT_LIMIT,
)

pd.DataFrame({
    "rank": np.arange(
        1,
        len(tree_consensus) + 1,
    ),
    "feature": tree_consensus,
}).to_csv(
    CHECKPOINT_DIR
    / "tree__consensus.csv",
    index=False,
)


tree_routes = {
    **tree_base_routes,
    "consensus": tree_consensus,
}


print(
    "Tree routes:",
    list(tree_routes),
)

[CACHE] tree__relevance
[CACHE] tree__model_gain
[CACHE] tree__temporal_relevance
[CACHE] tree__interaction_relevance
[CACHE] tree__redundancy_aware
Tree routes: ['relevance', 'model_gain', 'temporal_relevance', 'interaction_relevance', 'redundancy_aware', 'consensus']


## 9. Logistic Regression 分支 — 模型相关的排序策略

Tree branch 的所有 Evidence 不应该原封不动地拿给 Logistic Regression。

例如 LightGBM Gain 和 Tree-path Interaction，本质上描述的是 Tree learner 怎样利用 feature。

一个 feature 可能因为明显的非线性切分而非常适合 Tree，但标准 Logistic Regression 如果没有额外 transformation，并不一定同样受益。

因此 LR branch 先保留三个更通用的 Ranking Perspectives：

- **Relevance**：feature 自己有没有预测信息？
- **Temporal Relevance**：这种关系是否跨时间可靠？
- **Redundancy-aware**：前部是否被大量相似信息占满？

然后再生成 LR Consensus。

这里不是说 Tree-specific evidence 对 LR 完全没有参考价值，而是：

> **不同 learner 可以有不同的搜索 priority。**

最终哪条 LR path 更好，仍然要由 Notebook 10 的 LR wrapper evaluation 判断。


In [33]:
# ============================================================
# 9. Logistic Regression Ranking Routes
# ============================================================

lr_relevance = load_or_build_route(
    "lr__relevance",
    lambda: complete_order(
        selection_table
        .sort_values(
            [
                "relevance_score",
                "feature",
            ],
            ascending=[
                False,
                True,
            ],
            na_position="last",
        )[
            "feature"
        ]
        .tolist()
    ),
)


lr_temporal = load_or_build_route(
    "lr__temporal_relevance",
    lambda: complete_order(
        selection_table
        .sort_values(
            [
                "temporal_stable_flag",
                "temporal_gini_min",
                "relevance_score",
                "feature",
            ],
            ascending=[
                False,
                False,
                False,
                True,
            ],
            na_position="last",
        )[
            "feature"
        ]
        .tolist()
    ),
)


lr_redundancy = load_or_build_route(
    "lr__redundancy_aware",
    lambda: redundancy_aware_order(
        lr_relevance
    ),
)


lr_base_routes = {
    "relevance": lr_relevance,
    "temporal_relevance": lr_temporal,
    "redundancy_aware": lr_redundancy,
}


lr_consensus = consensus_order(
    lr_base_routes,
    CONSENSUS_FRONT_LIMIT,
)

pd.DataFrame({
    "rank": np.arange(
        1,
        len(lr_consensus) + 1,
    ),
    "feature": lr_consensus,
}).to_csv(
    CHECKPOINT_DIR
    / "lr__consensus.csv",
    index=False,
)


lr_routes = {
    **lr_base_routes,
    "lr_consensus": lr_consensus,
}


print(
    "LR routes:",
    list(lr_routes),
)

[CACHE] lr__relevance
[CACHE] lr__temporal_relevance
[CACHE] lr__redundancy_aware
LR routes: ['relevance', 'temporal_relevance', 'redundancy_aware', 'lr_consensus']


## 10. 敏感性变体的路线筛选

第 8、9 节已经生成了 **Base Route Families**。

现在第 5 节的 Ranking Robustness 副线重新回到主线。

核心问题是：

> **某个 sensitivity variant 只是“同一条路的小抖动”，还是已经形成值得 NB10 单独搜索的新方向？**

例如 Relevance family 可以暂时产生：

```text
50/50 reference
60/40
40/60
IV-only
MI-only
```

但这绝不意味着五条都要送进 Notebook 10。

因为：

> **便宜地产生一个 ranking，不代表它值得让 NB10 为它训练几十个 models。**

所以这里做的是“Route Promotion”，不是继续无限扩张 Route。

判断标准仍然是：

```text
整体 rank 是否明显变化？
Top200 / 300 / ... / 800 成员是否明显变化？
变化的 features 有没有明确方法意义？
```

如果没有新增 search direction：

> 不 Promote。

如果确实把一批不同而有意义的 features 带进前部：

> 才 Promote 少量有代表性的 variant。

这就是主线和副线真正会合的位置：

```text
主线：
构造少数 Search Routes

副线：
检查定义是否脆弱、避免重复成本

        ↓ 会合

最终只把“有信息增量”的 routes
交给 Notebook 10
```

### 情况 A：几乎完全重复

如果 IV-only 与 50/50 reference：

- rank correlation 很高；
- Top200–800 Jaccard 都很高；
- boundary 附近很少换人；

那么把 IV-only 再送进 NB10，只是在重复搜索。

这种成本应该砍掉。

### 情况 B：明显改变 Top-k composition

如果 MI-only 在多个 k 上都引入一批不同 features，那么它可能代表一个真正不同的 Relevance perspective。

这时多付一条 route 的 NB10 成本是有理由的，因为它在防止：

> **默认 50/50 definition 把搜索方向过早锁死。**

### 为什么不自动设一个 Jaccard threshold 决定晋升？

因为：

```text
Jaccard = 0.90
```

本身也是一个新的任意 threshold。

所以本版只输出 diagnostics，不自动决定。

NB8 正式重跑以后，再根据：

- overlap；
- rank shifts；
- 哪些 features 跨越 NB10 搜索边界；
- 业务 / temporal / interaction evidence；

决定是否晋升。

代码顶部预留：

```python
PROMOTE_TREE_RELEVANCE_VARIANTS
PROMOTE_LR_RELEVANCE_VARIANTS
```

默认都是空集合。

这意味着：

> **在看到真实 NB8 结果以前，不提前扩大昂贵搜索空间。**

---

### 本次 NB8 后的具体规则

`iv60_mi40` 与 `iv40_mi60` 只用于检查：

> **50/50 附近的小扰动是否已经足以改变 ranking。**

它们默认不允许直接 Promote。

如果：

```text
40/60
50/50
60/40
```

都很接近，就说明 default weighting 不脆弱。

如果连这种小扰动都明显换人，也避免把三条全部送进 NB10；应只选少数能代表结构差异的路线。

Interaction 同理：

```text
300 / 500 / 800
```

先比较 exclusive rescue set。

- 几乎不变 → 保留 300，砍掉后续成本；
- 500 / 800 新增很多真正 exclusive rescue features → 才考虑扩大 formal interaction limit。

Numerical redundancy 不需要 sensitivity route；这里修的是实现本身：

> **cluster 应按 group 去重优先级，而不是依赖任意 anchor。**


In [34]:
# ============================================================
# 10. Optional / Promoted Routes
# ============================================================
#
# 两种 route 来源：
#
# A) Sensitivity variant
#    已经在本册前面便宜地审查过；
#    只有确认“真正改变搜索方向”才晋升。
#
# B) Future evidence combination
#    有明确方法学理由时才新增。
#
# 默认全部为空，不扩大 NB10 搜索成本。


# ------------------------------------------------------------
# A. Promote relevance sensitivity variants
# ------------------------------------------------------------

ALLOWED_RELEVANCE_PROMOTIONS = {
    "iv_only",
    "mi_only",
}

unknown_tree_promotions = (
    set(
        PROMOTE_TREE_RELEVANCE_VARIANTS
    )
    - ALLOWED_RELEVANCE_PROMOTIONS
)

unknown_lr_promotions = (
    set(
        PROMOTE_LR_RELEVANCE_VARIANTS
    )
    - ALLOWED_RELEVANCE_PROMOTIONS
)

if unknown_tree_promotions:
    raise ValueError(
        "Unknown Tree sensitivity variants: "
        f"{sorted(unknown_tree_promotions)}"
    )

if unknown_lr_promotions:
    raise ValueError(
        "Unknown LR sensitivity variants: "
        f"{sorted(unknown_lr_promotions)}"
    )


for variant_name in (
    PROMOTE_TREE_RELEVANCE_VARIANTS
):
    tree_routes[
        f"relevance_{variant_name}"
    ] = load_or_build_route(
        (
            "tree__relevance_"
            f"{variant_name}"
        ),
        lambda variant_name=variant_name: (
            relevance_sensitivity_orders[
                variant_name
            ]
        ),
    )


for variant_name in (
    PROMOTE_LR_RELEVANCE_VARIANTS
):
    lr_routes[
        f"relevance_{variant_name}"
    ] = load_or_build_route(
        (
            "lr__relevance_"
            f"{variant_name}"
        ),
        lambda variant_name=variant_name: (
            relevance_sensitivity_orders[
                variant_name
            ]
        ),
    )


# ------------------------------------------------------------
# B. Optional future evidence-combination routes
# ------------------------------------------------------------

EXTRA_TREE_ROUTES = {}
EXTRA_LR_ROUTES = {}


def build_extra_route(
    columns,
    ascending,
):
    return complete_order(
        selection_table
        .sort_values(
            columns,
            ascending=ascending,
            na_position="last",
        )[
            "feature"
        ]
        .tolist()
    )


for route_name, config in (
    EXTRA_TREE_ROUTES.items()
):
    tree_routes[
        route_name
    ] = load_or_build_route(
        f"tree__{route_name}",
        lambda config=config: (
            build_extra_route(
                config["columns"],
                config["ascending"],
            )
        ),
    )


for route_name, config in (
    EXTRA_LR_ROUTES.items()
):
    lr_routes[
        route_name
    ] = load_or_build_route(
        f"lr__{route_name}",
        lambda config=config: (
            build_extra_route(
                config["columns"],
                config["ascending"],
            )
        ),
    )


# ------------------------------------------------------------
# Consensus must reflect the current formal route set.
# ------------------------------------------------------------

tree_consensus_base = {
    route_name: order
    for route_name, order
    in tree_routes.items()
    if route_name
    != "consensus"
}

tree_routes[
    "consensus"
] = consensus_order(
    tree_consensus_base,
    CONSENSUS_FRONT_LIMIT,
)


lr_consensus_base = {
    route_name: order
    for route_name, order
    in lr_routes.items()
    if route_name
    != "lr_consensus"
}

lr_routes[
    "lr_consensus"
] = consensus_order(
    lr_consensus_base,
    CONSENSUS_FRONT_LIMIT,
)


print(
    "Tree formal routes:",
    list(tree_routes),
)

print(
    "LR formal routes:",
    list(lr_routes),
)


Tree formal routes: ['relevance', 'model_gain', 'temporal_relevance', 'interaction_relevance', 'redundancy_aware', 'consensus']
LR formal routes: ['relevance', 'temporal_relevance', 'redundancy_aware', 'lr_consensus']


## 11. 正式路线的重叠度审查

不同 route 的名字不同，并不代表它们在探索不同 subset space。

例如：

```text
Relevance Route
Temporal Route
```

理论依据不同，但如果：

```text
Top200
Top300
...
Top800
```

几乎都选到同一批 features，那么让 NB10 对两条 route 各训练一整套 models，信息增量就很低。

因此这里不再只看一个 `Top300` overlap，而是和 NB10 的搜索深度对齐，在多个 k 上检查 Jaccard。

为什么要多 k？

因为两条 route 可能：

```text
Top200 很不同
但 Top800 几乎相同
```

也可能反过来。

而 NB10 真正搜索的是一串不同深度，所以重复程度也应该沿搜索深度观察。

### 成本判断

这个 audit 很便宜，却可以回答：

> **多付一整条 NB10 route 的成本，是在探索新区域，还是重复已有区域？**

但它仍然不自动删除 route。

因为某些很小的 membership difference 可能恰好集中在重要的 interaction / temporal features 上。

所以 Overlap 是诊断，不是新的 hard filter。


In [35]:
# ============================================================
# 11. Formal Route Preview + Multi-k Overlap Audit
# ============================================================

PREVIEW_N = 20


tree_preview = pd.DataFrame({
    route_name: order[
        :PREVIEW_N
    ]
    for route_name, order
    in tree_routes.items()
})


lr_preview = pd.DataFrame({
    route_name: order[
        :PREVIEW_N
    ]
    for route_name, order
    in lr_routes.items()
})


print(
    "Tree ranking preview"
)

display(
    tree_preview
)


print(
    "LR ranking preview"
)

display(
    lr_preview
)


def route_overlap_table(
    routes,
    k,
):
    route_names = list(
        routes
    )

    rows = []

    for index, route_a in enumerate(
        route_names
    ):
        for route_b in route_names[
            index + 1:
        ]:

            effective_k = min(
                k,
                len(routes[route_a]),
                len(routes[route_b]),
            )

            set_a = set(
                routes[
                    route_a
                ][
                    :effective_k
                ]
            )

            set_b = set(
                routes[
                    route_b
                ][
                    :effective_k
                ]
            )

            union = (
                set_a
                | set_b
            )

            rows.append({
                "route_a": route_a,
                "route_b": route_b,
                "k": effective_k,
                "overlap_count": len(
                    set_a
                    & set_b
                ),
                "jaccard": (
                    len(
                        set_a
                        & set_b
                    )
                    / len(union)
                    if union
                    else np.nan
                ),
            })

    return pd.DataFrame(
        rows
    )


tree_overlap = pd.concat(
    [
        route_overlap_table(
            tree_routes,
            k,
        )
        for k in RANKING_STABILITY_KS
        if k <= len(candidate_features)
    ],
    ignore_index=True,
)


lr_overlap = pd.concat(
    [
        route_overlap_table(
            lr_routes,
            k,
        )
        for k in RANKING_STABILITY_KS
        if k <= len(candidate_features)
    ],
    ignore_index=True,
)


print(
    "Tree multi-k overlap"
)

display(
    tree_overlap
    .sort_values(
        [
            "k",
            "jaccard",
        ],
        ascending=[
            True,
            False,
        ],
    )
)


print(
    "LR multi-k overlap"
)

display(
    lr_overlap
    .sort_values(
        [
            "k",
            "jaccard",
        ],
        ascending=[
            True,
            False,
        ],
    )
)


Tree ranking preview


,relevance,model_gain,temporal_relevance,interaction_relevance,redundancy_aware,consensus
0,datelastinstal40dpd_247D__days_before_decision,avgdpdtolclosure24_3658938P,pctinstlsallpaidlate1d_3546856L,applicationscnt_867L,datelastinstal40dpd_247D__days_before_decision,pctinstlsallpaidlate1d_3546856L
1,pmtcount_693L,person_main_registaddr_district_1083M,pmts_dpd_1073P_mean,collater_valueofguarantee_1124L_sum,pmtcount_693L,pctinstlsallpaidearl3d_427L
2,overdueamountmax2date_1142D_min__days_before_d...,person_main_contaddr_district_15M,pmts_overdue_1140A_mean,deposit_amount_mean,overdueamountmax2date_1142D_min__days_before_d...,lastrejectdate_50D__days_before_decision
3,assignmentdate_238D__days_before_decision,previouscontdistrict_112M,overdueamountmax_155A_mean,isbidproduct_1095L,assignmentdate_238D__days_before_decision,avgdpdtolclosure24_3658938P
4,numberofoverdueinstlmaxdat_641D_max__days_befo...,pmts_dpd_1073P_mean,dpdmax_139P_mean,clientscnt6m_3712949L,numberofoverdueinstlmaxdat_641D_max__days_befo...,maxdpdtolerance_577P_mean
5,overdueamountmax2date_1142D_max__days_before_d...,dpdmaxdateyear_596T_mean,maxdpdtolerance_577P_mean,collater_valueofguarantee_1124L_max,dpdmaxdateyear_596T_max,dpdmax_139P_max
6,dpdmaxdateyear_596T_max,pmts_overdue_1140A_mean,numberofoverdueinstlmax_1039L_mean,applicationscnt_1086L,lastdelinqdate_224D__days_before_decision,status_219L_K_pct
7,lastdelinqdate_224D__days_before_decision,maxdpdtolerance_577P_mean,overdueamountmax_155A_max,person_related1_type_25L,dpdmaxdateyear_896T_max,pmts_overdue_1140A_mean
8,dpdmaxdateyear_896T_max,person_main_empladdr_district_926M,pmts_overdue_1140A_max,currdebt_22A,numberofoverdueinstlmaxdat_641D_min__days_befo...,status_219L_D_pct
9,numberofoverdueinstlmaxdat_641D_min__days_befo...,pmts_dpd_303P_mean,pctinstlsallpaidlate4d_3546849L,instlamount_852A_mean,overdueamountmax2date_1002D_max__days_before_d...,pctinstlsallpaidlate6d_3546844L


LR ranking preview


,relevance,temporal_relevance,redundancy_aware,lr_consensus
0,datelastinstal40dpd_247D__days_before_decision,pctinstlsallpaidlate1d_3546856L,datelastinstal40dpd_247D__days_before_decision,datelastinstal40dpd_247D__days_before_decision
1,pmtcount_693L,pmts_dpd_1073P_mean,pmtcount_693L,overdueamountmax2date_1142D_min__days_before_d...
2,overdueamountmax2date_1142D_min__days_before_d...,pmts_overdue_1140A_mean,overdueamountmax2date_1142D_min__days_before_d...,numberofoverdueinstlmaxdat_641D_max__days_befo...
3,assignmentdate_238D__days_before_decision,overdueamountmax_155A_mean,assignmentdate_238D__days_before_decision,lastdelinqdate_224D__days_before_decision
4,numberofoverdueinstlmaxdat_641D_max__days_befo...,dpdmax_139P_mean,numberofoverdueinstlmaxdat_641D_max__days_befo...,dpdmaxdateyear_896T_max
5,overdueamountmax2date_1142D_max__days_before_d...,maxdpdtolerance_577P_mean,dpdmaxdateyear_596T_max,numberofoverdueinstlmaxdat_641D_min__days_befo...
6,dpdmaxdateyear_596T_max,numberofoverdueinstlmax_1039L_mean,lastdelinqdate_224D__days_before_decision,overdueamountmax2date_1002D_max__days_before_d...
7,lastdelinqdate_224D__days_before_decision,overdueamountmax_155A_max,dpdmaxdateyear_896T_max,rejectreason_755M_nunique
8,dpdmaxdateyear_896T_max,pmts_overdue_1140A_max,numberofoverdueinstlmaxdat_641D_min__days_befo...,pctinstlsallpaidlate1d_3546856L
9,numberofoverdueinstlmaxdat_641D_min__days_befo...,pctinstlsallpaidlate4d_3546849L,overdueamountmax2date_1002D_max__days_before_d...,numberofoverdueinstlmaxdat_148D_min__days_befo...


Tree multi-k overlap


,route_a,route_b,k,overlap_count,jaccard
2,relevance,interaction_relevance,200,162,0.680672
11,temporal_relevance,consensus,200,147,0.581028
4,relevance,consensus,200,142,0.550388
1,relevance,temporal_relevance,200,113,0.393728
13,interaction_relevance,consensus,200,113,0.393728
...,...,...,...,...,...
102,interaction_relevance,redundancy_aware,800,625,0.641026
95,model_gain,temporal_relevance,800,612,0.619433
99,temporal_relevance,interaction_relevance,800,609,0.614531
91,relevance,temporal_relevance,800,608,0.612903


LR multi-k overlap


,route_a,route_b,k,overlap_count,jaccard
4,temporal_relevance,lr_consensus,200,150,0.600000
2,relevance,lr_consensus,200,140,0.538462
5,redundancy_aware,lr_consensus,200,117,0.413428
0,relevance,temporal_relevance,200,113,0.393728
1,relevance,redundancy_aware,200,104,0.351351
3,temporal_relevance,redundancy_aware,200,76,0.234568
10,temporal_relevance,lr_consensus,300,230,0.621622
8,relevance,lr_consensus,300,201,0.503759
11,redundancy_aware,lr_consensus,300,187,0.452785
6,relevance,temporal_relevance,300,158,0.357466


## 12. 下游子集搜索的计算预算

Notebook 9 到这里完成的是：

> **从无结构的巨大 subset space，构造出少数值得继续探索的 priority paths。**

但它仍然没有回答：

1. 沿一条 path 应该走到多少 features？
2. 当前搜索边界够不够？
3. 固定 ranking 在局部有没有排错成员？

这些必须由 Notebook 10 用真实 learner performance 回答。

因此 NB10 的额外计算成本不是“重复 NB9”，而是在补三个不同的搜索盲点：

```text
Coarse Search
→ 防止只测试少数 k，造成 under-search

Boundary Check
→ 防止最佳点恰好落在人工搜索边界

Fine Search ±50
→ 防止 100-feature 网格过粗

Restricted Local Repair
→ 防止只改变 k、却永远不能修复 subset composition

F1–F4
→ 防止当前平均表现掩盖 temporal degradation
```

这些都是**有明确问题对象的必要成本**。

但 NB10 也不会把这些成本平均撒给所有候选：

```text
所有 routes
→ coarse

有竞争力的区域
→ fine

少数 finalists
→ local repair + full temporal evaluation
```

也就是说：

> **昂贵计算越往后，候选越少；但每一步付出的额外成本，都必须对应一个上一层无法回答的问题。**

Notebook 9 本身仍然不会输出最终 `selected_features.json`。

它输出完整 Ranking Paths、Sensitivity diagnostics 和 route-overlap information，交给 Notebook 10。


In [24]:
# ============================================================
# 12. Save Notebook 9 Final Outputs
# ============================================================

def routes_to_long(
    routes,
    branch,
):
    frames = []

    for route_name, order in (
        routes.items()
    ):

        frames.append(
            pd.DataFrame({
                "branch": branch,
                "route": route_name,
                "rank": np.arange(
                    1,
                    len(order) + 1,
                ),
                "feature": order,
            })
        )

    return pd.concat(
        frames,
        ignore_index=True,
    )


tree_long = routes_to_long(
    tree_routes,
    "tree",
)

lr_long = routes_to_long(
    lr_routes,
    "lr",
)


tree_long.to_csv(
    FINAL_DIR
    / "tree_rankings_long.csv",
    index=False,
)

lr_long.to_csv(
    FINAL_DIR
    / "lr_rankings_long.csv",
    index=False,
)

tree_overlap.to_csv(
    FINAL_DIR
    / "tree_route_overlap.csv",
    index=False,
)

lr_overlap.to_csv(
    FINAL_DIR
    / "lr_route_overlap.csv",
    index=False,
)


relevance_sensitivity_summary.to_csv(
    FINAL_DIR
    / "relevance_ranking_sensitivity.csv",
    index=False,
)

relevance_sensitivity_scores.to_csv(
    FINAL_DIR
    / "relevance_sensitivity_scores.csv",
    index=False,
)

model_gain_permutation_agreement.to_csv(
    FINAL_DIR
    / "gain_permutation_agreement.csv",
    index=False,
)


relevance_membership_changes.to_csv(
    FINAL_DIR
    / "relevance_membership_changes_vs_50_50.csv",
    index=False,
)

gain_permutation_disagreement.to_csv(
    FINAL_DIR
    / "gain_permutation_rank_disagreement.csv",
    index=False,
)

interaction_pair_limit_sensitivity.to_csv(
    FINAL_DIR
    / "interaction_pair_limit_sensitivity.csv",
    index=False,
)

interaction_rescue_recalibration.to_csv(
    FINAL_DIR
    / "interaction_rescue_recalibration.csv",
    index=False,
)

temporal_boundary_diagnostic.to_csv(
    FINAL_DIR
    / "temporal_boundary_diagnostic.csv",
    index=False,
)

with open(
    FINAL_DIR
    / "relevance_sensitivity_rankings.json",
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        relevance_sensitivity_orders,
        file,
        ensure_ascii=False,
        indent=2,
    )


with open(
    FINAL_DIR
    / "tree_rankings.json",
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        tree_routes,
        file,
        ensure_ascii=False,
        indent=2,
    )


with open(
    FINAL_DIR
    / "lr_rankings.json",
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        lr_routes,
        file,
        ensure_ascii=False,
        indent=2,
    )


nb9_config = {
    "version": (
        "ranking_routes_v2_dual_line"
    ),

    "candidate_feature_count": (
        len(candidate_features)
    ),


    "nb8_source_fingerprint": (
        NB8_SOURCE_FINGERPRINT
    ),

    "methodology": {
        "main_line": (
            "search-space construction"
        ),
        "side_line": (
            "ranking sensitivity / robustness"
        ),
        "cost_principle": (
            "increase cost only to address a defined "
            "search blind spot; remove low-information "
            "duplicate computation"
        ),
    },

    "relevance_sensitivity_weights": (
        RELEVANCE_SENSITIVITY_WEIGHTS
    ),

    "ranking_stability_ks": (
        RANKING_STABILITY_KS
    ),


    "interaction_pair_limit_sensitivity": (
        INTERACTION_PAIR_LIMIT_SENSITIVITY
    ),

    "interaction_formal_pair_limit": (
        INTERACTION_FORMAL_PAIR_LIMIT
    ),

    "temporal_gini_boundary_band": (
        TEMPORAL_GINI_BOUNDARY_BAND
    ),

    "numeric_redundancy_handling": (
        "cluster_membership_first; "
        "categorical/mixed pair graph second"
    ),

    "promoted_tree_relevance_variants": (
        sorted(
            PROMOTE_TREE_RELEVANCE_VARIANTS
        )
    ),

    "promoted_lr_relevance_variants": (
        sorted(
            PROMOTE_LR_RELEVANCE_VARIANTS
        )
    ),

    "tree_routes": list(
        tree_routes
    ),

    "lr_routes": list(
        lr_routes
    ),

    "consensus_front_limit": (
        CONSENSUS_FRONT_LIMIT
    ),

    "validation_read": False,
    "oot_read": False,

    "model_fitting_in_nb9": False,
    "subset_search_in_nb9": False,
    "hyperparameter_tuning_in_nb9": False,
}


with open(
    FINAL_DIR
    / "nb9_ranking_config.json",
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        nb9_config,
        file,
        ensure_ascii=False,
        indent=2,
    )


print(
    "Notebook 9 outputs:"
)

for path in sorted(
    FINAL_DIR.iterdir()
):
    print(
        " -",
        path.name,
    )

Notebook 9 outputs:
 - gain_permutation_agreement.csv
 - gain_permutation_rank_disagreement.csv
 - interaction_pair_limit_sensitivity.csv
 - interaction_rescue_recalibration.csv
 - lr_rankings.json
 - lr_rankings_long.csv
 - lr_route_overlap.csv
 - nb9_ranking_config.json
 - relevance_membership_changes_vs_50_50.csv
 - relevance_ranking_sensitivity.csv
 - relevance_sensitivity_rankings.json
 - relevance_sensitivity_scores.csv
 - temporal_boundary_diagnostic.csv
 - tree_rankings.json
 - tree_rankings_long.csv
 - tree_route_overlap.csv


## 13. 结果总结与后续接口

Notebook 09 输出完整的 Tree / LR ranking routes、参数敏感性诊断和跨路线 Top-k overlap 结果。相关检查用于评估不同 evidence weighting、interaction 定义和 temporal boundaries 是否改变候选特征集。

本阶段尚未确定最终特征数量或最终模型性能。Notebook 10 将沿这些路线执行粗到细的模型评估，并通过时间折及局部替换检查子集结果的稳定性。

```text
NB08: Evidence Construction
     ↓
NB09: Route Construction & Ranking Robustness
     ↓
NB10: Subset Evaluation & Temporal Diagnostics
```
